# [SQL 실습 #08] MySQL 정렬과 개수 제한 더 깊게 보기
## ORDER BY, LIMIT, OFFSET

> **학생용 실습 노트북 - TODO + 자동점검형**

이번 실습은 제공된 **SQL-8.pdf**의 흐름을 따라 구성했습니다.  
지난 시간까지 만든 `school.students` 데이터를 이용하여 **원하는 순서로 정렬하고, 필요한 개수만 조회하고, 앞의 데이터를 건너뛰는 방법**을 연습합니다.

---

## 오늘의 학습 목표

실습을 마치면 다음을 할 수 있어야 합니다.

1. `ORDER BY`의 역할을 설명할 수 있다.
2. `ASC`, `DESC`를 사용하여 오름차순/내림차순 정렬할 수 있다.
3. 여러 컬럼을 기준으로 정렬할 수 있다.
4. 컬럼마다 서로 다른 정렬 방향을 지정할 수 있다.
5. `LIMIT`으로 조회할 데이터 개수를 제한할 수 있다.
6. `ORDER BY`와 `LIMIT`을 함께 사용할 수 있다.
7. `OFFSET`으로 앞의 데이터를 건너뛸 수 있다.
8. `LIMIT + OFFSET`으로 게시판 페이지처럼 데이터를 나눠 볼 수 있다.
9. 페이지 번호와 `OFFSET`의 관계를 설명할 수 있다.
10. `LIMIT 개수 OFFSET 건너뛸개수`와 `LIMIT 건너뛸개수, 개수` 형식을 구분할 수 있다.

### 오늘의 핵심

> **ORDER BY는 줄 세우기, LIMIT는 필요한 만큼 꺼내기, OFFSET은 앞의 몇 줄을 건너뛰기입니다.**

# 1. MySQL 실습 환경 준비

Google Colab 런타임이 초기화되면 MySQL 서버와 이전 데이터가 사라질 수 있습니다.  
아래 셀에서 MySQL을 설치하고 실행합니다.

> 이 셀은 수정하지 말고 실행하세요.

In [ ]:
!sudo apt-get -qq update
!sudo DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mysql-server > /dev/null
!sudo service mysql start

print("✅ MySQL 서버 준비 완료")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
 * Starting MySQL database server mysqld        su: warning: cannot change directory to /nonexistent: No such file or directory
[ OK ]
✅ MySQL 서버 준비 완료


# 2. `school.students` 실습 데이터 준비

SQL-8 자료의 실습 흐름에 맞추어 **12명의 학생 데이터**를 준비합니다.

| id | name | grade | class_name |
|---:|---|---:|---|
| 1 | 김부장 | 3 | 컴퓨터과 |
| 2 | 혼이 | 1 | 영혼반 |
| 3 | 라즈베리 | 2 | 임베디드반 |
| 4 | 리눅스 | 3 | 서버반 |
| 5 | 파이썬 | 1 | 프로그래밍반 |
| 6 | 김코딩 | 1 | 컴퓨터과 |
| 7 | 데이터왕 | 2 | 데이터베이스반 |
| 8 | 홍길동 | 2 | 컴퓨터과 |
| 9 | 이몽룡 | 3 | 데이터베이스반 |
| 10 | 정렬이 | 1 | SQL반 |
| 11 | 제한이 | 2 | SQL반 |
| 12 | 페이징 | 3 | 웹개발반 |

> 아래 셀은 수정하지 말고 실행하세요.

In [ ]:
%%bash
sudo mysql <<'SQL'
CREATE DATABASE IF NOT EXISTS school
DEFAULT CHARACTER SET utf8mb4
DEFAULT COLLATE utf8mb4_unicode_ci;

USE school;

DROP TABLE IF EXISTS students;

CREATE TABLE students (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(50) NOT NULL,
    grade INT,
    class_name VARCHAR(50),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO students (name, grade, class_name) VALUES
('김부장', 3, '컴퓨터과'),
('혼이', 1, '영혼반'),
('라즈베리', 2, '임베디드반'),
('리눅스', 3, '서버반'),
('파이썬', 1, '프로그래밍반'),
('김코딩', 1, '컴퓨터과'),
('데이터왕', 2, '데이터베이스반'),
('홍길동', 2, '컴퓨터과'),
('이몽룡', 3, '데이터베이스반'),
('정렬이', 1, 'SQL반'),
('제한이', 2, 'SQL반'),
('페이징', 3, '웹개발반');

SELECT id, name, grade, class_name
FROM students
ORDER BY id;
SQL

id	name	grade	class_name
1	김부장	3	컴퓨터과
2	혼이	1	영혼반
3	라즈베리	2	임베디드반
4	리눅스	3	서버반
5	파이썬	1	프로그래밍반
6	김코딩	1	컴퓨터과
7	데이터왕	2	데이터베이스반
8	홍길동	2	컴퓨터과
9	이몽룡	3	데이터베이스반
10	정렬이	1	SQL반
11	제한이	2	SQL반
12	페이징	3	웹개발반


# 3. SQL 실행 / 자동점검 도우미

이 노트북의 TODO는 모두 **조회(SELECT) 실습**입니다.

- SQL에 `TODO`가 남아 있으면 실행하지 않습니다.
- `INSERT`, `UPDATE`, `DELETE`, `DROP`, `TRUNCATE`, `ALTER`는 학생 TODO에서 차단합니다.
- 자동점검은 **조회 결과의 값과 순서**를 함께 비교합니다.
- 정렬 실습이므로 같은 행이 나오더라도 순서가 다르면 통과하지 않을 수 있습니다.

> 이 셀은 수정하지 마세요.

In [ ]:
import subprocess
import textwrap
import re

TODO_STATUS = {}
LAST_SQL = ""

def _mysql(sql, no_header=False):
    cmd = ["sudo", "mysql", "--batch", "--raw"]
    if no_header:
        cmd.append("-N")
    result = subprocess.run(
        cmd,
        input=textwrap.dedent(sql).strip(),
        text=True,
        capture_output=True
    )
    return result.returncode == 0, result.stdout.strip(), result.stderr.strip()

def _real(sql):
    return "\n".join(
        line for line in textwrap.dedent(sql).splitlines()
        if line.strip() and not line.lstrip().startswith("--")
    ).strip()

def run_sql(sql):
    global LAST_SQL
    LAST_SQL = sql
    real = _real(sql)

    if not real:
        print("⚠️ SQL이 비어 있습니다.")
        return False

    if "TODO" in real:
        print("⚠️ SQL 안에 TODO가 남아 있습니다. 먼저 완성하세요.")
        return False

    if re.search(r"\b(INSERT|UPDATE|DELETE|DROP|TRUNCATE|ALTER|CREATE)\b", real, re.I):
        print("❌ 이 실습의 TODO 셀은 조회 전용입니다.")
        return False

    ok, out, err = _mysql(sql)
    if out:
        print(out)

    if not ok:
        print("❌ MySQL 오류")
        print(err)
        return False

    print("✅ 조회 완료")
    return True

def _result(sql):
    ok, out, err = _mysql(sql, no_header=True)
    if not ok:
        raise RuntimeError(err)
    return out.splitlines() if out else []

def check_todo(todo_id, student_sql, expected_sql, required_patterns=()):
    real = _real(student_sql)

    if not real or "TODO" in real:
        TODO_STATUS[todo_id] = False
        print(f"🔎 TODO {todo_id} 자동점검")
        print("❌ TODO를 먼저 완성하세요.")
        return False

    for p in required_patterns:
        if not re.search(p, real, re.I | re.S):
            TODO_STATUS[todo_id] = False
            print(f"🔎 TODO {todo_id} 자동점검")
            print("❌ 필요한 SQL 문법이 빠졌습니다.")
            return False

    try:
        actual = _result(student_sql)
        expected = _result(expected_sql)
    except Exception as e:
        TODO_STATUS[todo_id] = False
        print(f"🔎 TODO {todo_id} 자동점검")
        print("❌ SQL 실행 오류:", e)
        return False

    passed = (actual == expected)
    TODO_STATUS[todo_id] = passed

    print(f"🔎 TODO {todo_id} 자동점검")
    if passed:
        print("✅ 통과했습니다.")
    else:
        print("❌ 결과 또는 정렬 순서를 다시 확인하세요.")
        print("예상 행 수:", len(expected), "/ 현재 행 수:", len(actual))
    return passed

def show_progress(total):
    print("===== TODO 진행 현황 =====")
    passed = 0
    for i in range(1, total + 1):
        state = TODO_STATUS.get(i)
        mark = "✅" if state is True else ("❌" if state is False else "⬜")
        if state is True:
            passed += 1
        print(f"{mark} TODO {i}")
    print(f"\n진행률: {passed}/{total}")

# 4. 사용할 데이터베이스 확인하기

먼저 `school` 데이터베이스를 선택하고 `students` 테이블이 있는지 확인합니다.

## TODO 1

아래 SQL의 `TODO`를 완성하세요.

힌트:

```sql
USE 데이터베이스명;
SHOW TABLES;
```

In [ ]:
sql = '''
USE school;
SHOW tables;
'''
_todo_id = 1
run_sql(sql)

Tables_in_school
students
✅ 조회 완료


True

In [ ]:
# TODO 1 자동점검 - 수정하지 마세요.
check_todo(
    1, sql,
    "USE school; SHOW TABLES;",
    (r"\bUSE\s+SCHOOL\b", r"\bSHOW\s+TABLES\b")
)

🔎 TODO 1 자동점검
✅ 통과했습니다.


True

# 5. 전체 데이터 확인하기

정렬을 배우기 전에 현재 데이터를 `id` 순으로 확인합니다.

## TODO 2

`students` 테이블의 `id`, `name`, `grade`, `class_name`을 조회하고  
`id`를 **오름차순**으로 정렬하세요.

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY id ASC;
'''
_todo_id = 2
run_sql(sql)

id	name	grade	class_name
1	김부장	3	컴퓨터과
2	혼이	1	영혼반
3	라즈베리	2	임베디드반
4	리눅스	3	서버반
5	파이썬	1	프로그래밍반
6	김코딩	1	컴퓨터과
7	데이터왕	2	데이터베이스반
8	홍길동	2	컴퓨터과
9	이몽룡	3	데이터베이스반
10	정렬이	1	SQL반
11	제한이	2	SQL반
12	페이징	3	웹개발반
✅ 조회 완료


True

In [ ]:
check_todo(
    2, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY id ASC;",
    (r"\bFROM\s+STUDENTS\b", r"\bORDER\s+BY\s+ID\b")
)

🔎 TODO 2 자동점검
✅ 통과했습니다.


True

# 6. ORDER BY란?

`ORDER BY`는 **조회 결과의 순서를 정하는 문법**입니다.

기본 형식:

```sql
SELECT 컬럼명
FROM 테이블명
ORDER BY 정렬기준;
```

예를 들어 학년을 기준으로 정렬하려면:

```sql
SELECT id, name, grade, class_name
FROM students
ORDER BY grade;
```

`ASC`를 생략하면 기본적으로 오름차순으로 정렬됩니다.

## TODO 3

학생을 `grade` 기준으로 정렬하세요.  
이번에는 `ASC`를 적지 말고 기본 정렬을 확인합니다.

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY grade;
'''
_todo_id = 3
run_sql(sql)

id	name	grade	class_name
2	혼이	1	영혼반
5	파이썬	1	프로그래밍반
6	김코딩	1	컴퓨터과
10	정렬이	1	SQL반
3	라즈베리	2	임베디드반
7	데이터왕	2	데이터베이스반
8	홍길동	2	컴퓨터과
11	제한이	2	SQL반
1	김부장	3	컴퓨터과
4	리눅스	3	서버반
9	이몽룡	3	데이터베이스반
12	페이징	3	웹개발반
✅ 조회 완료


True

In [ ]:
check_todo(
    3, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY grade;",
    (r"\bORDER\s+BY\s+GRADE\b",)
)

🔎 TODO 3 자동점검
✅ 통과했습니다.


True

# 7. ASC 오름차순 정렬

`ASC`는 **ascending**, 즉 오름차순입니다.

- 숫자: 작은 값 → 큰 값
- 문자: 앞 순서 → 뒤 순서
- 학년: 1학년 → 2학년 → 3학년

> **낮은 번호부터 차례대로 줄 세우는 것**이라고 생각하면 쉽습니다.

## TODO 4

학생을 `grade` 오름차순으로 정렬하세요.

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY grade ASC;
'''
_todo_id = 4
run_sql(sql)

id	name	grade	class_name
2	혼이	1	영혼반
5	파이썬	1	프로그래밍반
6	김코딩	1	컴퓨터과
10	정렬이	1	SQL반
3	라즈베리	2	임베디드반
7	데이터왕	2	데이터베이스반
8	홍길동	2	컴퓨터과
11	제한이	2	SQL반
1	김부장	3	컴퓨터과
4	리눅스	3	서버반
9	이몽룡	3	데이터베이스반
12	페이징	3	웹개발반
✅ 조회 완료


True

In [ ]:
check_todo(
    4, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY grade ASC;",
    (r"\bORDER\s+BY\s+GRADE\s+ASC\b",)
)

🔎 TODO 4 자동점검
✅ 통과했습니다.


True

# 8. DESC 내림차순 정렬

`DESC`는 **descending**, 즉 내림차순입니다.

- 숫자: 큰 값 → 작은 값
- 문자: 뒤 순서 → 앞 순서
- 학년: 3학년 → 2학년 → 1학년

> **높은 번호부터 거꾸로 줄 세우는 것**이라고 생각하면 쉽습니다.

## TODO 5

학생을 `grade` 내림차순으로 정렬하세요.

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY grade DESC;
'''
_todo_id = 5
run_sql(sql)

id	name	grade	class_name
1	김부장	3	컴퓨터과
4	리눅스	3	서버반
9	이몽룡	3	데이터베이스반
12	페이징	3	웹개발반
3	라즈베리	2	임베디드반
7	데이터왕	2	데이터베이스반
8	홍길동	2	컴퓨터과
11	제한이	2	SQL반
2	혼이	1	영혼반
5	파이썬	1	프로그래밍반
6	김코딩	1	컴퓨터과
10	정렬이	1	SQL반
✅ 조회 완료


True

In [ ]:
check_todo(
    5, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY grade DESC;",
    (r"\bORDER\s+BY\s+GRADE\s+DESC\b",)
)

🔎 TODO 5 자동점검
✅ 통과했습니다.


True

# 9. 여러 기준으로 정렬하기

정렬 기준은 하나만 사용할 필요가 없습니다.

```sql
ORDER BY grade ASC, name ASC;
```

의미:

1. 먼저 `grade`로 정렬
2. 같은 학년 안에서는 `name`으로 다시 정렬

> **먼저 큰 기준으로 줄을 세우고, 같은 줄 안에서 작은 기준으로 다시 정리합니다.**

## TODO 6

`grade` 오름차순 → 같은 학년에서는 `name` 오름차순으로 정렬하세요.

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY grade ASC, name ASC;
'''
_todo_id = 6
run_sql(sql)

id	name	grade	class_name
6	김코딩	1	컴퓨터과
10	정렬이	1	SQL반
5	파이썬	1	프로그래밍반
2	혼이	1	영혼반
7	데이터왕	2	데이터베이스반
3	라즈베리	2	임베디드반
11	제한이	2	SQL반
8	홍길동	2	컴퓨터과
1	김부장	3	컴퓨터과
4	리눅스	3	서버반
9	이몽룡	3	데이터베이스반
12	페이징	3	웹개발반
✅ 조회 완료


True

In [ ]:
check_todo(
    6, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY grade ASC, name ASC;",
    (r"\bORDER\s+BY\s+GRADE\s+ASC\s*,\s*NAME\s+ASC\b",)
)

🔎 TODO 6 자동점검
✅ 통과했습니다.


True

# 10. 서로 다른 방향으로 정렬하기

여러 정렬 기준마다 방향을 다르게 지정할 수도 있습니다.

```sql
ORDER BY grade DESC, name ASC;
```

의미:

1. 학년은 높은 학년부터
2. 같은 학년 안에서는 이름을 오름차순으로

## TODO 7

`grade`는 내림차순, 같은 학년에서는 `name`을 오름차순으로 정렬하세요.

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY grade DESC, name ASC;
'''
_todo_id = 7
run_sql(sql)

id	name	grade	class_name
1	김부장	3	컴퓨터과
4	리눅스	3	서버반
9	이몽룡	3	데이터베이스반
12	페이징	3	웹개발반
7	데이터왕	2	데이터베이스반
3	라즈베리	2	임베디드반
11	제한이	2	SQL반
8	홍길동	2	컴퓨터과
6	김코딩	1	컴퓨터과
10	정렬이	1	SQL반
5	파이썬	1	프로그래밍반
2	혼이	1	영혼반
✅ 조회 완료


True

In [ ]:
check_todo(
    7, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY grade DESC, name ASC;",
    (r"\bORDER\s+BY\s+GRADE\s+DESC\s*,\s*NAME\s+ASC\b",)
)

🔎 TODO 7 자동점검
✅ 통과했습니다.


True

# 11. 최근 데이터부터 보기

`AUTO_INCREMENT`로 만들어진 `id`는 보통 나중에 입력된 데이터일수록 큰 값을 가집니다.

따라서 아래처럼 정렬하면 **최근에 입력된 데이터부터** 볼 수 있습니다.

```sql
ORDER BY id DESC;
```

## TODO 8

`id`가 큰 학생부터 조회하세요.

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY id DESC;
'''
_todo_id = 8
run_sql(sql)

id	name	grade	class_name
12	페이징	3	웹개발반
11	제한이	2	SQL반
10	정렬이	1	SQL반
9	이몽룡	3	데이터베이스반
8	홍길동	2	컴퓨터과
7	데이터왕	2	데이터베이스반
6	김코딩	1	컴퓨터과
5	파이썬	1	프로그래밍반
4	리눅스	3	서버반
3	라즈베리	2	임베디드반
2	혼이	1	영혼반
1	김부장	3	컴퓨터과
✅ 조회 완료


True

In [ ]:
check_todo(
    8, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY id DESC;",
    (r"\bORDER\s+BY\s+ID\s+DESC\b",)
)

🔎 TODO 8 자동점검
✅ 통과했습니다.


True

# 12. LIMIT이란?

`LIMIT`는 **조회할 데이터의 개수를 제한**할 때 사용합니다.

기본 형식:

```sql
SELECT 컬럼명
FROM 테이블명
LIMIT 개수;
```

예:

```sql
LIMIT 3;
```

→ 조회 결과에서 **3개만 가져옵니다.**

> `LIMIT`는 서류 뭉치를 전부 펼쳐보는 것이 아니라 **앞에서 필요한 개수만 꺼내 보는 것**입니다.

## TODO 9

`id` 오름차순으로 정렬한 뒤 **앞의 3명만** 조회하세요.

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY id ASC
LIMIT 3;
'''
_todo_id = 9
run_sql(sql)

id	name	grade	class_name
1	김부장	3	컴퓨터과
2	혼이	1	영혼반
3	라즈베리	2	임베디드반
✅ 조회 완료


True

In [ ]:
check_todo(
    9, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY id ASC LIMIT 3;",
    (r"\bLIMIT\s+3\b",)
)

🔎 TODO 9 자동점검
✅ 통과했습니다.


True

# 13. ORDER BY와 LIMIT 함께 사용하기

`LIMIT`만 사용하면 어떤 데이터가 먼저 나올지 의도와 다를 수 있습니다.  
따라서 **정렬한 뒤 필요한 개수만 가져오는 습관**이 중요합니다.

SQL 작성 순서:

```sql
SELECT ...
FROM ...
ORDER BY ...
LIMIT ...;
```

## TODO 10

최근에 입력된 학생 3명을 조회하세요.

조건:

- `id` 내림차순
- 3명만 조회

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY id DESC
LIMIT 3;
'''
_todo_id = 10
run_sql(sql)

id	name	grade	class_name
12	페이징	3	웹개발반
11	제한이	2	SQL반
10	정렬이	1	SQL반
✅ 조회 완료


True

In [ ]:
check_todo(
    10, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY id DESC LIMIT 3;",
    (r"\bORDER\s+BY\s+ID\s+DESC\b", r"\bLIMIT\s+3\b")
)

🔎 TODO 10 자동점검
✅ 통과했습니다.


True

# 14. OFFSET이란?

`OFFSET`은 **앞의 데이터를 몇 개 건너뛸지** 지정합니다.

기본 형식:

```sql
LIMIT 가져올개수 OFFSET 건너뛸개수;
```

예:

```sql
LIMIT 3 OFFSET 3;
```

의미:

- 앞의 3개를 건너뛰고
- 그 다음 3개를 가져옵니다.

즉, `id` 오름차순이라면 4번째, 5번째, 6번째 데이터가 조회됩니다.

## TODO 11

`id` 오름차순으로 정렬한 뒤 **앞의 3명을 건너뛰고 다음 3명**을 조회하세요.

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY id ASC
LIMIT 3 OFFSET 3;
'''
_todo_id = 11
run_sql(sql)

id	name	grade	class_name
4	리눅스	3	서버반
5	파이썬	1	프로그래밍반
6	김코딩	1	컴퓨터과
✅ 조회 완료


True

In [ ]:
check_todo(
    11, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY id ASC LIMIT 3 OFFSET 3;",
    (r"\bLIMIT\s+3\s+OFFSET\s+3\b",)
)

🔎 TODO 11 자동점검
✅ 통과했습니다.


True

# 15. 게시판 페이지처럼 나눠 보기

한 페이지에 5개씩 보여준다고 생각해 봅시다.

### 1페이지
- 건너뛰기: 0개
- 가져오기: 5개

```sql
LIMIT 5 OFFSET 0;
```

## TODO 12

1페이지 데이터를 조회하세요.

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY id ASC
LIMIT 5 OFFSET 0;
'''
_todo_id = 12
run_sql(sql)

id	name	grade	class_name
1	김부장	3	컴퓨터과
2	혼이	1	영혼반
3	라즈베리	2	임베디드반
4	리눅스	3	서버반
5	파이썬	1	프로그래밍반
✅ 조회 완료


True

In [ ]:
check_todo(
    12, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY id ASC LIMIT 5 OFFSET 0;",
    (r"\bLIMIT\s+5\s+OFFSET\s+0\b",)
)

🔎 TODO 12 자동점검
✅ 통과했습니다.


True

### 2페이지

- 앞의 5개를 건너뛰고
- 다음 5개를 가져옵니다.

## TODO 13

2페이지 데이터를 조회하세요.

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY id ASC
LIMIT 5 OFFSET 5;
'''
_todo_id = 13
run_sql(sql)

id	name	grade	class_name
6	김코딩	1	컴퓨터과
7	데이터왕	2	데이터베이스반
8	홍길동	2	컴퓨터과
9	이몽룡	3	데이터베이스반
10	정렬이	1	SQL반
✅ 조회 완료


True

In [ ]:
check_todo(
    13, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY id ASC LIMIT 5 OFFSET 5;",
    (r"\bLIMIT\s+5\s+OFFSET\s+5\b",)
)

🔎 TODO 13 자동점검
✅ 통과했습니다.


True

### 3페이지

- 앞의 10개를 건너뛰고
- 다음 5개를 가져옵니다.

현재 데이터는 12개이므로 실제로는 남은 데이터만 조회됩니다.

## TODO 14

3페이지 데이터를 조회하세요.

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY id ASC
LIMIT 5 OFFSET 10;
'''
_todo_id = 14
run_sql(sql)

id	name	grade	class_name
11	제한이	2	SQL반
12	페이징	3	웹개발반
✅ 조회 완료


True

In [ ]:
check_todo(
    14, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY id ASC LIMIT 5 OFFSET 10;",
    (r"\bLIMIT\s+5\s+OFFSET\s+10\b",)
)

🔎 TODO 14 자동점검
✅ 통과했습니다.


True

# 16. 페이지 번호와 OFFSET 계산하기

한 페이지에 보여 줄 데이터 개수를 `페이지당 개수`라고 하면:

```text
OFFSET = (페이지번호 - 1) × 페이지당 개수
```

예를 들어 한 페이지에 5개씩 보여줄 때:

| 페이지 | OFFSET |
|---:|---:|
| 1 | 0 |
| 2 | 5 |
| 3 | 10 |
| 4 | 15 |

## TODO 15

**4페이지**, 페이지당 **5개**라고 가정합니다.  
파이썬 코드를 완성하여 OFFSET을 계산하세요.

In [ ]:
page = 4
page_size = 5

offset = (page - 1) * page_size

print("페이지:", page)
print("페이지당 개수:", page_size)
print("OFFSET:", offset)

# 정답은 OFFSET 15가 되어야 합니다.

페이지: 4
페이지당 개수: 5
OFFSET: 15


In [ ]:
# TODO 15 자동점검 - 수정하지 마세요.
try:
    passed = (page == 4 and page_size == 5 and offset == 15)
except Exception:
    passed = False

TODO_STATUS[15] = passed
print("🔎 TODO 15 자동점검")
print("✅ 통과했습니다." if passed else "❌ page=4, page_size=5가 되도록 완성하세요.")

🔎 TODO 15 자동점검
✅ 통과했습니다.


# 17. LIMIT의 다른 문법

MySQL에서는 다음 두 문장이 같은 의미입니다.

```sql
LIMIT 3 OFFSET 6;
```

```sql
LIMIT 6, 3;
```

두 번째 형식은:

```text
LIMIT 건너뛸개수, 가져올개수
```

입니다.

> 순서가 반대이므로 헷갈리지 않도록 주의합니다.

## TODO 16

`LIMIT 건너뛸개수, 가져올개수` 형식을 사용하여  
앞의 3개를 건너뛰고 다음 3개를 조회하세요.

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY id ASC
LIMIT 3, 3;
'''
_todo_id = 16
run_sql(sql)

id	name	grade	class_name
4	리눅스	3	서버반
5	파이썬	1	프로그래밍반
6	김코딩	1	컴퓨터과
✅ 조회 완료


True

In [ ]:
check_todo(
    16, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY id ASC LIMIT 3, 3;",
    (r"\bLIMIT\s+3\s*,\s*3\b",)
)

🔎 TODO 16 자동점검
✅ 통과했습니다.


True

# 18. 상위 몇 개만 조회하기

정렬과 `LIMIT`을 함께 사용하면 **상위 N개**를 쉽게 조회할 수 있습니다.

예를 들어 학년이 높은 학생부터 보고 싶다면:

```sql
ORDER BY grade DESC
LIMIT 3;
```

같은 학년이 여러 명이면 결과 순서를 더 안정적으로 만들기 위해 두 번째 기준을 추가할 수 있습니다.

## TODO 17

조건:

1. `grade`가 높은 학생부터
2. 같은 학년에서는 `id`가 작은 학생부터
3. 상위 3명만 조회

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY grade DESC, id ASC
LIMIT 3;
'''
_todo_id = 17
run_sql(sql)

id	name	grade	class_name
1	김부장	3	컴퓨터과
4	리눅스	3	서버반
9	이몽룡	3	데이터베이스반
✅ 조회 완료


True

In [ ]:
check_todo(
    17, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY grade DESC, id ASC LIMIT 3;",
    (r"\bORDER\s+BY\s+GRADE\s+DESC\s*,\s*ID\s+ASC\b", r"\bLIMIT\s+3\b")
)

🔎 TODO 17 자동점검
✅ 통과했습니다.


True

# 19. 최신 데이터 몇 개만 조회하기

게시판의 최신 글, 최근 가입자, 최근 등록 데이터처럼  
**가장 최근 데이터 몇 개만** 보고 싶은 경우 자주 사용하는 형태입니다.

```sql
ORDER BY id DESC
LIMIT 5;
```

## TODO 18

가장 최근에 입력된 학생 **5명**을 조회하세요.

In [ ]:
sql = '''
USE school;

SELECT id, name, grade, class_name
FROM students
ORDER BY id DESC
LIMIT 5;
'''
_todo_id = 18
run_sql(sql)

id	name	grade	class_name
12	페이징	3	웹개발반
11	제한이	2	SQL반
10	정렬이	1	SQL반
9	이몽룡	3	데이터베이스반
8	홍길동	2	컴퓨터과
✅ 조회 완료


True

In [ ]:
check_todo(
    18, sql,
    "USE school; SELECT id,name,grade,class_name FROM students ORDER BY id DESC LIMIT 5;",
    (r"\bORDER\s+BY\s+ID\s+DESC\b", r"\bLIMIT\s+5\b")
)

🔎 TODO 18 자동점검
✅ 통과했습니다.


True

# 20. 정렬 기준이 없으면 결과 순서를 믿지 않기

다음처럼 `ORDER BY` 없이 `LIMIT`만 작성할 수도 있습니다.

```sql
SELECT id, name, grade, class_name
FROM students
LIMIT 5;
```

하지만 **어떤 5개가 어떤 순서로 나올지 항상 보장되는 것은 아닙니다.**

원하는 기준이 있다면 반드시 정렬 기준을 명확하게 작성합니다.

```sql
SELECT id, name, grade, class_name
FROM students
ORDER BY id ASC
LIMIT 5;
```

> **앞에서 5줄을 가져오라고 하기 전에, 먼저 어떤 순서로 줄을 세울지 정해야 합니다.**

# 21. 자주 발생하는 실수

## 1) ORDER BY 위치를 잘못 쓰는 경우

잘못된 예:

```sql
SELECT id, name, grade, class_name
FROM students
LIMIT 5
ORDER BY id DESC;
```

올바른 순서:

```sql
SELECT ...
FROM ...
ORDER BY ...
LIMIT ...;
```

---

## 2) LIMIT과 ORDER BY의 순서를 바꾸는 경우

SQL 문장의 기본 순서를 기억하세요.

```text
SELECT
FROM
WHERE
ORDER BY
LIMIT
```

---

## 3) OFFSET만 단독으로 사용하는 경우

MySQL에서는 보통 다음처럼 `LIMIT`과 함께 사용합니다.

```sql
LIMIT 5 OFFSET 5;
```

---

## 4) 페이지 계산을 헷갈리는 경우

페이지가 10개씩이고 3페이지라면:

```text
OFFSET = (3 - 1) × 10 = 20
```

따라서:

```sql
LIMIT 10 OFFSET 20;
```

# 22. 종합 실습

아래 문제는 자동점검 TODO에는 포함하지 않습니다.  
수업 시간에 짝 활동 또는 마무리 문제로 활용하세요.

### 문제 A
2학년 학생을 이름 오름차순으로 조회하세요.

### 문제 B
학년이 높은 순서로 정렬하고, 같은 학년에서는 이름 오름차순으로 정렬하세요.

### 문제 C
`id`가 큰 순서로 정렬한 뒤 최근 학생 4명만 조회하세요.

### 문제 D
한 페이지에 4명씩 보여줄 때 2페이지를 조회하는 SQL을 작성하세요.

<details>
<summary>정답 예시 보기</summary>

```sql
-- A
SELECT id, name, grade, class_name
FROM students
WHERE grade = 2
ORDER BY name ASC;

-- B
SELECT id, name, grade, class_name
FROM students
ORDER BY grade DESC, name ASC;

-- C
SELECT id, name, grade, class_name
FROM students
ORDER BY id DESC
LIMIT 4;

-- D
SELECT id, name, grade, class_name
FROM students
ORDER BY id ASC
LIMIT 4 OFFSET 4;
```

</details>

# 23. 오늘 실습 코드 전체 정리

```sql
USE school;

-- 오름차순
SELECT id, name, grade, class_name
FROM students
ORDER BY grade ASC;

-- 내림차순
SELECT id, name, grade, class_name
FROM students
ORDER BY grade DESC;

-- 여러 기준
SELECT id, name, grade, class_name
FROM students
ORDER BY grade ASC, name ASC;

-- 서로 다른 방향
SELECT id, name, grade, class_name
FROM students
ORDER BY grade DESC, name ASC;

-- 최근 데이터부터
SELECT id, name, grade, class_name
FROM students
ORDER BY id DESC;

-- 앞에서 3개
SELECT id, name, grade, class_name
FROM students
ORDER BY id ASC
LIMIT 3;

-- 최근 3개
SELECT id, name, grade, class_name
FROM students
ORDER BY id DESC
LIMIT 3;

-- 앞의 3개를 건너뛰고 3개
SELECT id, name, grade, class_name
FROM students
ORDER BY id ASC
LIMIT 3 OFFSET 3;

-- 한 페이지 5개, 2페이지
SELECT id, name, grade, class_name
FROM students
ORDER BY id ASC
LIMIT 5 OFFSET 5;

-- MySQL의 다른 LIMIT 문법
SELECT id, name, grade, class_name
FROM students
ORDER BY id ASC
LIMIT 3, 3;
```

# 24. 전체 TODO 진행 현황

아래 셀에서 TODO 1~18의 자동점검 결과를 확인하세요.

- ✅ : 통과
- ❌ : 재확인 필요
- ⬜ : 아직 점검하지 않음

> 런타임을 다시 시작하면 진행 상태도 초기화됩니다.

In [ ]:
show_progress(18)

===== TODO 진행 현황 =====
✅ TODO 1
✅ TODO 2
✅ TODO 3
✅ TODO 4
✅ TODO 5
✅ TODO 6
✅ TODO 7
✅ TODO 8
✅ TODO 9
✅ TODO 10
✅ TODO 11
✅ TODO 12
✅ TODO 13
✅ TODO 14
✅ TODO 15
✅ TODO 16
✅ TODO 17
✅ TODO 18

진행률: 18/18


# 25. 오늘 배운 내용 정리

| 문법 | 역할 |
|---|---|
| `ORDER BY` | 조회 결과의 순서를 정함 |
| `ASC` | 오름차순 |
| `DESC` | 내림차순 |
| `LIMIT n` | 최대 n개만 조회 |
| `OFFSET n` | 앞의 n개를 건너뜀 |
| `LIMIT n OFFSET m` | m개를 건너뛰고 n개 조회 |
| `LIMIT m, n` | MySQL 형식: m개를 건너뛰고 n개 조회 |

### 꼭 기억할 점

- `ORDER BY`는 **줄 세우기**
- `LIMIT`는 **필요한 만큼 꺼내기**
- `OFFSET`은 **앞의 몇 줄을 건너뛰기**
- 여러 정렬 기준은 왼쪽부터 우선 적용된다.
- `ASC`는 생략할 수 있지만 명확하게 적어도 좋다.
- `LIMIT`을 사용할 때 원하는 순서가 있다면 `ORDER BY`를 함께 사용하는 것이 좋다.
- 페이지 번호가 1부터 시작할 때:

```text
OFFSET = (페이지번호 - 1) × 페이지당 개수
```

> **원하는 순서로, 필요한 개수만, 정확한 위치에서 가져오는 것**이 오늘의 핵심입니다.

# 26. 자기 점검

- [ ] `ORDER BY`의 역할을 설명할 수 있다.
- [ ] `ASC`와 `DESC`의 차이를 설명할 수 있다.
- [ ] 한 개의 컬럼으로 정렬할 수 있다.
- [ ] 여러 컬럼으로 정렬할 수 있다.
- [ ] 정렬 기준마다 서로 다른 방향을 지정할 수 있다.
- [ ] 최근 입력 데이터부터 조회할 수 있다.
- [ ] `LIMIT`으로 조회 개수를 제한할 수 있다.
- [ ] `ORDER BY + LIMIT`을 함께 사용할 수 있다.
- [ ] `OFFSET`의 역할을 설명할 수 있다.
- [ ] 게시판처럼 페이지별 데이터를 조회할 수 있다.
- [ ] 페이지 번호로 OFFSET을 계산할 수 있다.
- [ ] `LIMIT n OFFSET m`과 `LIMIT m, n`의 차이를 설명할 수 있다.
- [ ] `LIMIT`을 사용할 때 정렬 기준을 명확히 해야 하는 이유를 설명할 수 있다.

# 다음 시간 예고

## [SQL 실습 #09] MySQL 집계 함수 기초
### COUNT, SUM, AVG, MAX, MIN

다음 시간에는 데이터를 단순히 조회하는 것을 넘어  
**학생 수, 합계, 평균, 가장 큰 값, 가장 작은 값**을 계산하는 방법을 배웁니다.